# Rank and expert sweep validity check

This notebook checks whether the rank and expert sweep reports are valid enough to use for conclusions. The Excel reports are the source of truth for selected benchmark rows; source artifacts on the anonymous-cluster cluster are used to verify checkpoints, final eval loss, target modules, and training exposure.

First we define criteria for what each run should fulfill to be considered "valid".

Validity criteria:
- all approaches train on the same data consisting of 10 languages
- all approaches train for approx. 3.6h
- actual training-data exposure does not exceed one epoch
- benchmark result file exists and has all expected benchmark-family rows
- selected run is marked clean by the training pipeline source artifact check
- selected adapter/checkpoint artifact exists and is the adapter checkpoint created shortly before training finishes
- final eval loss is present when claimed; if absent, it is a recomputable caveat and not used for winner selection here
- LoRA target modules are full, or explicitly intended for MoV, otherwise flagged


In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "outputs").exists():
    ROOT = Path("/cluster/user/moe-study")

NOTEBOOK_DIR = ROOT / "result_analysis/notebooks/validation"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from util import (
    adapter_dir_from_result,
    as_path,
    load_training_metrics,
    norm_targets,
    read_json,
    result_path_kind,
    run_dir,
    selected_checkpoint_step,
    source_root_from_result,
)

REPORTS = {
    "rank": ROOT / "result_analysis/reports/moe_study/moe_study_rank_sweep.xlsx",
    "expert": ROOT / "result_analysis/reports/moe_study/moe_study_expert_sweep.xlsx",
}

FULL_TARGETS = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}
MOV_TARGETS = {"k_proj", "v_proj", "down_proj"}

for name, path in REPORTS.items():
    print(name, path, path.exists())


rank /cluster/user/moe-study/outputs/pipeline_runs/moe_study_rank_sweep_pipeline_report/rank_pipeline_report.xlsx True
expert /cluster/user/moe-study/outputs/pipeline_runs/moe_study_expert_sweep_pipeline_report/expert_pipeline_report.xlsx True


## Validation logic

Generic artifact helpers live in `util.py`. The validation-specific logic stays here so the criteria remain visible in the notebook. `trainer_state.max_steps` is always populated by the trainer, so repeated-data risk is decided from `trainer_state.num_train_epochs > 1`, while actual repeated exposure is decided from the final `epoch > 1`.


In [2]:
def target_policy(kind, targets, source_root):
    target_set = set(targets)
    root_name = source_root.name if source_root else ""
    if not targets:
        return "unknown", "target modules unavailable"
    if target_set == FULL_TARGETS:
        return "full", "full q/k/v/o + gate/up/down target set"
    if kind == "movlora" and target_set == MOV_TARGETS:
        return "intended_mov_core", "intended MoV core target set: k_proj,v_proj,down_proj"
    if "rescue_moelora_vonly" in root_name:
        return "reduced", "MoELoRA v_proj-only rescue configuration"
    if "rescue_kvdown" in root_name:
        return "reduced", "KV/down rescue configuration used for high-memory rank runs"
    if "failed_recovery_full" in root_name:
        return "reduced", "HMORA failed-recovery run selected KV/down adapters"
    if "attention" in root_name or "remaining_problematic_recovery" in root_name:
        return "reduced", "attention-only recovery/calibrated configuration"
    return "reduced", "non-full target set"


In [3]:
def audit_sweep(sweep, report_path):
    cfg = pd.read_excel(report_path, sheet_name="run_configs")
    raw = pd.read_excel(report_path, sheet_name="benchmark_raw")
    manifest = pd.read_excel(report_path, sheet_name="source_manifest")
    final_eval = pd.read_excel(report_path, sheet_name="final_eval_loss")

    cfg_by_run = {str(row.run_id): row._asdict() for row in cfg.itertuples(index=False)}
    fe_by_run = {str(row.run_id): row._asdict() for row in final_eval.itertuples(index=False)}
    selected = manifest[manifest["selected_for_main"].astype(str).str.lower().eq("yes")].copy()

    baseline_mask = raw["is_baseline"].fillna(False).astype(bool) if "is_baseline" in raw else pd.Series(False, index=raw.index)
    expected_rows = int(raw[~baseline_mask].groupby("run_id").size().max())
    expected_families = sorted(raw["benchmark"].dropna().astype(str).unique())
    rows = []

    for mrow in selected.itertuples(index=False):
        run_id = str(mrow.run_id)
        result_path = as_path(getattr(mrow, "result_path", None), ROOT)
        source_root = source_root_from_result(result_path)
        rd = run_dir(source_root, run_id)
        ad = adapter_dir_from_result(result_path, source_root, run_id)
        adapter_cfg = read_json(ad / "adapter_config.json") if ad else {}
        cfg_row = cfg_by_run.get(run_id, {})
        fe_row = fe_by_run.get(run_id, {})

        targets = norm_targets(adapter_cfg.get("target_modules")) or norm_targets(cfg_row.get("target_modules"))
        policy, reason = target_policy(str(mrow.kind), targets, source_root)
        family_counts = raw[raw["run_id"].astype(str).eq(run_id)].groupby("benchmark").size().to_dict()
        benchmark_rows = int(sum(family_counts.values()))
        benchmark_complete = benchmark_rows == expected_rows and all(fam in family_counts for fam in expected_families)

        row = {
            "sweep": sweep,
            "run_id": run_id,
            "kind": str(mrow.kind),
            "source_root": str(source_root) if source_root else "",
            "result_path": str(result_path) if result_path else "",
            "result_path_kind": result_path_kind(getattr(mrow, "result_path", None)),
            "result_exists": bool(result_path and result_path.exists()),
            "run_dir": str(rd) if rd else "",
            "adapter_dir": str(ad) if ad else "",
            "adapter_exists": bool(ad and ad.exists()),
            "adapter_config_exists": bool(adapter_cfg),
            "selected_checkpoint_step": selected_checkpoint_step(ad),
            "manifest_clean": str(mrow.status) == "complete_clean" and str(mrow.checkpoint_status) == "complete_clean",
            "benchmark_rows": benchmark_rows,
            "expected_benchmark_rows": expected_rows,
            "benchmark_complete": benchmark_complete,
            "benchmark_family_counts": json.dumps(family_counts, sort_keys=True),
            "final_eval_present": str(fe_row.get("final_eval_present", "")).lower() == "yes",
            "final_eval_loss": fe_row.get("eval_loss"),
            "final_eval_source_path": fe_row.get("source_path"),
            "target_modules": ",".join(targets),
            "target_module_policy": policy,
            "target_module_reason": reason,
            "max_steps_reported": cfg_row.get("max_steps"),
        }
        row.update(load_training_metrics(rd))
        rows.append(row)

    out = pd.DataFrame(rows)
    for col in ["epoch", "num_train_epochs", "global_step", "max_steps", "max_steps_reported", "num_input_tokens_seen", "train_runtime"]:
        if col in out:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    out["actual_repeat"] = out["epoch"] > 1.0001
    out["configured_repeat_risk"] = out["num_train_epochs"] > 1
    out["validity_notes"] = out.apply(validity_notes, axis=1)
    out["validity_status"] = np.where(out["validity_notes"].eq(""), "valid", "valid_with_note")
    return out


def validity_notes(row):
    notes = []
    for col, note in [
        ("manifest_clean", "manifest_not_clean"),
        ("result_exists", "result_missing"),
        ("benchmark_complete", "benchmark_incomplete"),
        ("adapter_config_exists", "adapter_config_missing"),
    ]:
        if not bool(row.get(col)):
            notes.append(note)
    if not bool(row.get("final_eval_present")):
        notes.append("final_eval_loss_missing")
    if row.get("result_path_kind") == "per_task_json":
        notes.append("per_task_json_benchmark_source")
    if row.get("target_module_policy") == "reduced":
        notes.append("reduced_target_modules")
    if bool(row.get("configured_repeat_risk")):
        notes.append("configured_repeat_risk")
    if bool(row.get("actual_repeat")):
        notes.append("actual_repeat_over_one_epoch")
    return ";".join(notes)

### Overview of rank and expert sweep runs


In [4]:
runs = pd.concat([audit_sweep(name, path) for name, path in REPORTS.items()], ignore_index=True)

overview = runs.groupby("sweep").agg(
    selected_runs=("run_id", "nunique"),
    complete_benchmarks=("benchmark_complete", "sum"),
    final_eval_present=("final_eval_present", "sum"),
    actual_repeat=("actual_repeat", "sum"),
    configured_repeat_risk=("configured_repeat_risk", "sum"),
    full_targets=("target_module_policy", lambda s: (s == "full").sum()),
    intended_mov_core=("target_module_policy", lambda s: (s == "intended_mov_core").sum()),
    reduced_targets=("target_module_policy", lambda s: (s == "reduced").sum()),
).reset_index()

display(overview)

,sweep,selected_runs,complete_benchmarks,final_eval_present,actual_repeat,configured_repeat_risk,full_targets,intended_mov_core,reduced_targets
0,expert,68,68,68,0,33,44,5,19
1,rank,55,55,0,0,0,36,1,18


### First validation check: data


In [5]:
# verify training data: same language set, and whether the exact tokenized dataset path is shared
DEFAULT_TOKENIZED = ROOT / "outputs/pipeline_runs/moe_study_profile_otus_cluster_10_langs/tokenized"
MOV_TOKENIZED = ROOT / "outputs/pipeline_runs/moe_study_movlora_dense_expert_sweep_sf0005/tokenized_clean"

def infer_tokenized_path(source_root):
    root = Path(source_root)
    profile = read_json(root / "state/resolved_profile.json")
    configured = profile.get("tokenization", {}).get("output_dir")
    if configured:
        path = Path(configured)
        return path if path.is_absolute() else ROOT / path
    if "movlora_dense" in root.name:
        return MOV_TOKENIZED
    return DEFAULT_TOKENIZED

def tokenized_meta(path):
    check = read_json(Path(path) / "_check.json")
    train_tokens = sum(check.get("tokens_per_language_by_split", {}).get("train", {}).values())
    return pd.Series({
        "tokenized_path": str(Path(path).relative_to(ROOT)),
        "num_language_ids": check.get("num_languages"),
        "languages": ",".join(sorted(check.get("samples_per_language", {}).keys())),
        "train_rows": check.get("splits", {}).get("train"),
        "train_tokens": train_tokens,
    })

data_check = runs[["sweep", "kind", "run_id", "source_root"]].copy()
data_check["tokenized_abs"] = data_check.source_root.map(infer_tokenized_path)
data_check = pd.concat([data_check, data_check.tokenized_abs.apply(tokenized_meta)], axis=1)
display(data_check.groupby(["sweep", "tokenized_path", "num_language_ids", "languages", "train_rows", "train_tokens"]).agg(
    runs=("run_id", "nunique"), kinds=("kind", lambda s: ",".join(sorted(set(s))))
).reset_index())

same_language_set = data_check.languages.nunique() == 1
same_exact_dataset = data_check.tokenized_path.nunique() == 1
mov_uses_separate_data = data_check.kind.eq("movlora").any() and data_check[data_check.kind.eq("movlora")].tokenized_path.nunique() == 1 and not same_exact_dataset
standard_rows = data_check[~data_check.kind.eq("movlora")].drop_duplicates("tokenized_path")
mov_rows = data_check[data_check.kind.eq("movlora")].drop_duplicates("tokenized_path")

lines = [
    f"**Training data verdict:** same language/script set = `{same_language_set}`; exact same tokenized dataset path = `{same_exact_dataset}`.",
    f"The metadata has `{data_check.num_language_ids.iloc[0]}` language/script IDs from the 10-language profile.",
]
if mov_uses_separate_data:
    mov_tokens = int(mov_rows.train_tokens.iloc[0]) if len(mov_rows) else 0
    base_tokens = int(standard_rows.train_tokens.iloc[0]) if len(standard_rows) else 0
    lines.append(f"MoV is the only separate data caveat: it uses the same language/script set but a larger retokenized pool (`{mov_tokens:,}` train tokens vs `{base_tokens:,}`).")

display(Markdown(" ".join(lines)))

,sweep,tokenized_path,num_language_ids,languages,train_rows,train_tokens,runs,kinds
0,expert,outputs/pipeline_runs/moe_study_movlora_dense_...,13,"arb_Arab,arb_Latn,azj_Cyrl,azj_Latn,cmn_Hani,e...",198533,203297792,5,movlora
1,expert,outputs/pipeline_runs/moe_study_profile_otus_c...,13,"arb_Arab,arb_Latn,azj_Cyrl,azj_Latn,cmn_Hani,e...",99410,101795840,63,"adamole,cola,hmora,hydralora,moelora,mola,mtllora"
2,rank,outputs/pipeline_runs/moe_study_profile_otus_c...,13,"arb_Arab,arb_Latn,azj_Cyrl,azj_Latn,cmn_Hani,e...",99410,101795840,55,"cola,hmora,hydralora,lora,moelora,mola,movlora..."


**Training data verdict:** same language/script set = `True`; exact same tokenized dataset path = `False`. The metadata has `13` language/script IDs from the 10-language profile.

All approaches are trained on the same 10 languages. We have 13 language IDs because some languages are represented with different scripts.

For the expert sweep, MoV needed a retokenized dataset because the dense MoV runs would otherwise run out of data before the 3.6h compute budget. This MoV dataset uses the same language/script set, same tokenization settings, same sample seed, and same data distribution, but a larger sample fraction. It is therefore the main data-lineage caveat: distributionally comparable, but not the exact same tokenized dataset path.

The apparent `doc_id` non-overlap is not a clean content-overlap check because `doc_id` includes the absolute shard path. The old dataset was tokenized from the shared `/cluster/user/.../sharded_samples` path, while MoV was tokenized from `outputs/sharded_samples`; identical text lines would therefore get different `doc_id` hashes.


### Next check: same training time / compute budget

This is important since we base the study on an equal compute-budget comparison.


In [6]:
# verify wall-clock training time: approximately 3.6h per selected run
runtime = runs.copy()
runtime["train_hours"] = runtime.train_runtime / 3600
runtime_summary = runtime.groupby(["sweep", "kind"]).agg(
    runs=("run_id", "nunique"),
    hours_min=("train_hours", "min"),
    hours_median=("train_hours", "median"),
    hours_max=("train_hours", "max"),
).reset_index()
display(runtime_summary)

overall_runtime = runtime.groupby("sweep").agg(
    hours_min=("train_hours", "min"),
    hours_median=("train_hours", "median"),
    hours_max=("train_hours", "max"),
).reset_index()
display(overall_runtime)
within_approx_36h = runtime.train_hours.between(3.25, 3.75).all()
display(Markdown(f"**Runtime verdict:** all selected runs are within 3.25-3.75h = `{within_approx_36h}`; medians are rank `{overall_runtime.loc[overall_runtime.sweep.eq('rank'), 'hours_median'].iloc[0]:.2f}h` and expert `{overall_runtime.loc[overall_runtime.sweep.eq('expert'), 'hours_median'].iloc[0]:.2f}h`."))

,sweep,kind,runs,hours_min,hours_median,hours_max
0,expert,adamole,4,3.554016,3.562277,3.569010
1,expert,cola,5,3.297677,3.383486,3.462189
2,expert,hmora,15,3.542447,3.546334,3.595978
3,expert,hydralora,5,3.528034,3.549164,3.606076
4,expert,moelora,14,3.541546,3.566363,3.572582
5,expert,mola,15,3.562736,3.586194,3.609809
6,expert,movlora,5,3.529296,3.547201,3.555733
7,expert,mtllora,5,3.339036,3.405884,3.492703
8,rank,cola,8,3.452139,3.457724,3.462351
9,rank,hmora,7,3.570808,3.576984,3.583843


,sweep,hours_min,hours_median,hours_max
0,expert,3.297677,3.561462,3.609809
1,rank,3.445759,3.543153,3.617686


**Runtime verdict:** all selected runs are within 3.25-3.75h = `True`; medians are rank `3.54h` and expert `3.56h`.

The variations are created due to FSDP checkpoint materialization and different required time for checkpoint saving, which can depend on the compute node, approach, and FSDP config. In the current setup, before the compute limit is achieved, a SLURM signal is injected into the Hugging Face Trainer process, which triggers an adapter checkpoint mechanism. Check `LLaMA-Factory/src/llamafactory/train/callbacks.py` for more details. This was implemented to ensure scalable experiments with the same compute-budget time for many approaches that heavily vary in throughput and training speed on the same data.


## Benchmark and final evaluation coverage

This checks if we have all benchmark metric results for FLORES / Belebele / SIB / Taxi for all trained runs.

Final eval loss is a separate, post-training stage. It is currently not used to decide which approach is better, since we rely on the benchmark metrics. Missing final eval loss is therefore a caveat, but it can be recomputed from the selected checkpoints if needed.


In [7]:
coverage = runs.groupby(["sweep", "kind"]).agg(
    runs=("run_id", "nunique"),
    benchmark_complete=("benchmark_complete", "sum"),
    final_eval_present=("final_eval_present", "sum"),
    result_path_kinds=("result_path_kind", lambda s: ",".join(sorted(set(s)))),
).reset_index()
display(coverage)

source_eval = []
for root_text, group in runs.groupby("source_root"):
    root = Path(root_text)
    final_eval_dir = root / "final_eval_loss"
    eval_json_count = len(list(final_eval_dir.glob("**/eval_results.json"))) if final_eval_dir.exists() else 0
    final_eval_logs = len(list((root / "logs").glob("**/*final_eval*"))) if (root / "logs").exists() else 0
    source_eval.append({
        "source_root": root.name,
        "sweep": ",".join(sorted(set(group.sweep))),
        "runs": group.run_id.nunique(),
        "final_eval_dir_exists": final_eval_dir.exists(),
        "eval_results_json_count": eval_json_count,
        "final_eval_log_count": final_eval_logs,
    })
display(pd.DataFrame(source_eval).sort_values(["sweep", "source_root"]))

,sweep,kind,runs,benchmark_complete,final_eval_present,result_path_kinds
0,expert,adamole,4,4,4,file
1,expert,cola,5,5,5,"file,per_task_json"
2,expert,hmora,15,15,15,file
3,expert,hydralora,5,5,5,file
4,expert,moelora,14,14,14,file
5,expert,mola,15,15,15,file
6,expert,movlora,5,5,5,file
7,expert,mtllora,5,5,5,per_task_json
8,rank,cola,8,8,0,file
9,rank,hmora,7,7,0,file


,source_root,sweep,runs,final_eval_dir_exists,eval_results_json_count,final_eval_log_count
0,moe_study_expert_head_sweep_r2_noctua2_a100_4g...,expert,30,True,0,295
1,moe_study_expert_head_sweep_r2_noctua2_a100_4g...,expert,11,True,0,55
2,moe_study_hmora_attention_calibrated,expert,6,True,0,24
3,moe_study_hydralora_heads32_rerun,expert,1,True,0,7
4,moe_study_moelora_attention_sizefsdp_calibrated,expert,11,True,0,26
5,moe_study_movlora_dense_expert_sweep_sf0005_ca...,expert,5,True,0,15
10,moe_study_remaining_problematic_recovery,expert,4,True,0,17
6,moe_study_rank_expert_sweep_noctua2_a100_4gpu_...,rank,37,False,0,1
7,moe_study_rank_expert_sweep_noctua2_a100_4gpu_...,rank,5,False,0,0
8,moe_study_rank_expert_sweep_noctua2_a100_4gpu_...,rank,6,False,0,0


### LoRA target module checks

We use this check because we decided to reduce the target modules for approaches where full LoRA target modules were infeasible to run under the GPU memory constraints. In those cases, we reduced to attention-only or another smaller target set.

MoV uses `k_proj,v_proj,down_proj` by design, so these runs are marked as `intended_mov_core`. Other non-full target sets are real caveats and are grouped below by reason.


In [8]:
target_counts = runs.groupby(["sweep", "target_module_policy"]).size().reset_index(name="runs")
display(target_counts)

reduced = runs[runs.target_module_policy.eq("reduced")]
reduced_summary = reduced.groupby(["sweep", "kind", "target_modules", "target_module_reason"]).agg(
    runs=("run_id", "nunique"),
    run_ids=("run_id", lambda s: ", ".join(s)),
).reset_index()
display(reduced_summary)

mov = runs[runs.target_module_policy.eq("intended_mov_core")][["sweep", "run_id", "kind", "target_modules"]]
display(mov)

,sweep,target_module_policy,runs
0,expert,full,44
1,expert,intended_mov_core,5
2,expert,reduced,19
3,rank,full,36
4,rank,intended_mov_core,1
5,rank,reduced,18


,sweep,kind,target_modules,target_module_reason,runs,run_ids
0,expert,adamole,"k_proj,o_proj,q_proj,v_proj",attention-only recovery/calibrated configuration,1,adamole_r2_e16
1,expert,hmora,"k_proj,o_proj,q_proj,v_proj",attention-only recovery/calibrated configuration,6,"hmora_r2_e2_k1, hmora_r2_e32_k1, hmora_r2_e32_..."
2,expert,moelora,"k_proj,o_proj,q_proj,v_proj",attention-only recovery/calibrated configuration,11,"moelora_r2_e8_k1, moelora_r2_e8_k2, moelora_r2..."
3,expert,mola,"k_proj,o_proj,q_proj,v_proj",attention-only recovery/calibrated configuration,1,adamoe_1_r2_e32_k16
4,rank,cola,"down_proj,k_proj,v_proj",KV/down rescue configuration used for high-mem...,1,cola_r256
5,rank,hmora,"down_proj,k_proj,v_proj",HMORA failed-recovery run selected KV/down ada...,7,"hmora_tr16_e8_k2, hmora_tr32_e8_k2, hmora_tr64..."
6,rank,hydralora,"down_proj,k_proj,v_proj",KV/down rescue configuration used for high-mem...,1,hydralora_r256
7,rank,lora,"down_proj,k_proj,v_proj",KV/down rescue configuration used for high-mem...,1,lora_r1024
8,rank,moelora,v_proj,MoELoRA v_proj-only rescue configuration,6,"moelora_tr16_e8_k2, moelora_tr32_e8_k2, moelor..."
9,rank,mola,"down_proj,k_proj,v_proj",KV/down rescue configuration used for high-mem...,1,mola_tr1024_e8_k6


,sweep,run_id,kind,target_modules
12,rank,movlora_e8,movlora,"down_proj,k_proj,v_proj"
118,expert,mov_dense_e2,movlora,"down_proj,k_proj,v_proj"
119,expert,mov_dense_e4,movlora,"down_proj,k_proj,v_proj"
120,expert,mov_dense_e8,movlora,"down_proj,k_proj,v_proj"
121,expert,mov_dense_e16,movlora,"down_proj,k_proj,v_proj"
122,expert,mov_dense_e32,movlora,"down_proj,k_proj,v_proj"


So in the expert experiments, 19 runs trained with reduced target modules. For the rank experiment, it was 18 runs. These are not hidden failures; they are memory-driven configuration caveats.


## Training exposure

This checks whether a run saw the data more than once, e.g. because the data was not enough and training repeated the epoch data.


In [9]:
exposure = runs.groupby(["sweep", "kind", "configured_repeat_risk", "actual_repeat"]).agg(
    runs=("run_id", "nunique"),
    epoch_min=("epoch", "min"),
    epoch_median=("epoch", "median"),
    epoch_max=("epoch", "max"),
    global_step_min=("global_step", "min"),
    global_step_max=("global_step", "max"),
    max_steps_values=("max_steps", lambda s: ",".join(map(str, sorted(set(s.dropna().astype(int)))))),
).reset_index()
display(exposure)

repeat_risk = runs[runs.configured_repeat_risk].copy()
display(repeat_risk[[
    "sweep", "run_id", "kind", "num_train_epochs", "epoch", "global_step", "max_steps", "source_root"
]].sort_values(["sweep", "kind", "run_id"]))

,sweep,kind,configured_repeat_risk,actual_repeat,runs,epoch_min,epoch_median,epoch_max,global_step_min,global_step_max,max_steps_values
0,expert,adamole,False,False,2,0.171408,0.214139,0.256870,355,532,2072
1,expert,adamole,True,False,2,0.094154,0.098982,0.103810,195,215,10000
2,expert,cola,False,False,5,0.911168,1.000000,1.000000,2831,3107,3107
3,expert,hmora,True,False,15,0.507947,0.517121,0.647461,1052,1341,10000
4,expert,hydralora,False,False,4,0.356804,0.581073,0.756578,739,1567,2072
5,expert,hydralora,True,False,1,0.201819,0.201819,0.201819,418,418,10000
6,expert,moelora,True,False,14,0.333642,0.627196,0.886940,691,1837,10000
7,expert,mola,False,False,14,0.169530,0.357927,0.748522,301,1329,1776
8,expert,mola,True,False,1,0.058424,0.058424,0.058424,121,121,10000
9,expert,movlora,False,False,5,0.552816,0.577955,0.590041,2287,2441,4137


,sweep,run_id,kind,num_train_epochs,epoch,global_step,max_steps,source_root
72,expert,adamole_r2_e16,adamole,5,0.103810,215,10000,/cluster/user/moe-study/outputs...
71,expert,adamole_r2_e8,adamole,5,0.094154,195,10000,/cluster/user/moe-study/outputs...
94,expert,hmora_r2_e16_k1,hmora,5,0.508912,1054,10000,/cluster/user/moe-study/outputs...
95,expert,hmora_r2_e16_k2,hmora,5,0.509878,1056,10000,/cluster/user/moe-study/outputs...
96,expert,hmora_r2_e16_k4,hmora,5,0.507947,1052,10000,/cluster/user/moe-study/outputs...
97,expert,hmora_r2_e16_k8,hmora,5,0.510361,1057,10000,/cluster/user/moe-study/outputs...
88,expert,hmora_r2_e2_k1,hmora,5,0.647461,1341,10000,/cluster/user/moe-study/outputs...
98,expert,hmora_r2_e32_k1,hmora,5,0.644081,1334,10000,/cluster/user/moe-study/outputs...
102,expert,hmora_r2_e32_k16,hmora,5,0.646013,1338,10000,/cluster/user/moe-study/outputs...
99,expert,hmora_r2_e32_k2,hmora,5,0.647461,1341,10000,/cluster/user/moe-study/outputs...


This shows:
- some runs of `mtllora` and `cola` in the expert sweep finished one entire epoch, but were calculated in a way that still kept them inside the intended max compute-budget constraint
- the bigger caveat is the MoV run family. MoV needed to be retokenized with more available data, since otherwise it would either have trained into a second epoch on the same data or stopped earlier. Therefore the MoV runs were stopped and rerun with a bigger sample using the same tokenization config, same data distribution, and same sample seed. This is not exactly the same tokenized data as before and should stay documented as a validity note.


In [10]:
def yes_no(value):
    return "yes" if bool(value) else "no"

rank = runs[runs.sweep.eq("rank")]
expert = runs[runs.sweep.eq("expert")]

rank_reduced = int((rank.target_module_policy == "reduced").sum())
expert_reduced = int((expert.target_module_policy == "reduced").sum())
expert_mov = int((expert.target_module_policy == "intended_mov_core").sum())
rank_missing_eval = rank.run_id.nunique() - int(rank.final_eval_present.sum())
expert_missing_eval = expert.run_id.nunique() - int(expert.final_eval_present.sum())
mov_data_caveat = data_check.kind.eq("movlora").any() and data_check.tokenized_path.nunique() > 1

lines = [
    "## Validity conclusion",
    "",
    f"- Rank sweep: {rank.run_id.nunique()}/{rank.run_id.nunique()} selected runs have complete benchmark rows.",
    f"- Expert sweep: {expert.run_id.nunique()}/{expert.run_id.nunique()} selected runs have complete benchmark rows.",
    f"- Repeated-data exposure: rank has {int(rank.actual_repeat.sum())} runs with `epoch > 1`; expert has {int(expert.actual_repeat.sum())} runs with `epoch > 1`.",
    f"- Final eval loss caveat: missing for {rank_missing_eval} rank runs and {expert_missing_eval} expert runs. This is not used for winner selection and can be recomputed from the selected checkpoints if needed.",
    f"- Target-module caveat: {rank_reduced} rank runs and {expert_reduced} expert runs use reduced non-MoV target modules, mostly because full target modules were infeasible under memory constraints.",
    f"- MoV caveat: {expert_mov} expert runs use the intended MoV core target set (`k_proj,v_proj,down_proj`). MoV also uses a larger retokenized dataset with the same language/script set and distribution = `{mov_data_caveat}`.",
    "",
    "Overall: the rank and expert sweeps are usable for benchmark-based conclusions, with the three documented caveats: MoV data lineage, recomputable missing final eval loss, and memory-driven reduced target modules.",
]
display(Markdown("\n".join(lines)))


## Validity conclusion

- Rank sweep: 55/55 selected runs have complete benchmark rows.
- Expert sweep: 68/68 selected runs have complete benchmark rows.
- Repeated-data exposure: rank has 0 runs with `epoch > 1`; expert has 0 runs with `epoch > 1`.
- Final eval loss caveat: missing for 55 rank runs and 0 expert runs. This is not used for winner selection and can be recomputed from the selected checkpoints if needed.
- Target-module caveat: 18 rank runs and 19 expert runs use reduced non-MoV target modules, mostly because full target modules were infeasible under memory constraints.
- MoV caveat: 5 expert runs use the intended MoV core target set (`k_proj,v_proj,down_proj`). MoV also uses a larger retokenized dataset with the same language/script set and distribution = `True`.

Overall: the rank and expert sweeps are usable for benchmark-based conclusions, with the three documented caveats: MoV data lineage, recomputable missing final eval loss, and memory-driven reduced target modules.